# Example 25: Interactive Workspace (topoview) Demo - 7D Torus

In this notebook, we showcase the `visualize_topoview` API, which provides a high-level interactive workspace for exploring the topological and geometric invariants of a simplicial complex.

We will:
1. Generate a **500-point 7-dimensional torus** ($T^2$) embedded in $\mathbb{R}^7$.
2. Construct its **simplicial complex** using the CkNN method.
3. **Calculate invariants** ($H_0, H_1, H_2, \pi_1$) beforehand.
4. Use `topoview` to visualize the workspace using these pre-calculated inputs.

In [1]:
import numpy as np
import pysurgery as ps
from pysurgery.topoview import visualize_topoview
from pysurgery.bridge.julia_bridge import julia_engine
from pysurgery.core.complexes import CWComplex
from time import time

if julia_engine.available:
    julia_engine.warmup()

# 1. Generate 100 points on a 2-torus in R7
n_points = 100
theta = np.random.uniform(0, 2 * np.pi, n_points)
phi = np.random.uniform(0, 2 * np.pi, n_points)
R, r = 2.0, 0.5

points_7d_torus = np.column_stack([
    (R + r * np.cos(theta)) * np.cos(phi),   # x1
    (R + r * np.cos(theta)) * np.sin(phi),   # x2
    r * np.sin(theta),                       # x3
    np.cos(theta),                           # x4
    np.sin(theta),                           # x5
    np.cos(phi),                             # x6
    np.sin(phi)                              # x7
])

print(f"Generated {n_points} points in 7D representing a torus.")

Detected IPython. Loading juliacall extension. See https://juliapy.github.io/PythonCall.jl/stable/compat/#IPython
Generated 100 points in 7D representing a torus.


In [2]:
# 2. Build the complex
sc = ps.SimplicialComplex.from_point_cloud_cknn(
    points_7d_torus, 
    k=15, 
    delta=1.2, 
    max_dimension=2
)

print(f"Simplicial Complex built.")

Simplicial Complex built.


## Calculating Invariants

We calculate the homology basis and fundamental group traces before passing them to the visualizer. This allows for more control over the computation parameters (e.g., using `optimal` mode for $H_1$).

In [ ]:
# 3. Calculate invariants
t0 = time()
h0 = ps.compute_homology_basis_from_complex(sc, dimension=0, mode='optimal')
print(f"H0 basis computed in {time() - t0:.2f} seconds.")
h1 = ps.compute_optimal_h1_basis_from_complex(sc, point_cloud=points_7d_torus)
print(f"H1 basis computed in {time() - t0:.2f} seconds.")
h2 = ps.compute_homology_basis_from_complex(sc, dimension=2, mode="optimal")
print(f"H2 basis computed in {time() - t0:.2f} seconds.")
pi1 = ps.extract_pi_1_with_traces(CWComplex.from_simplicial_complex(sc), simplify=True)
print(fr"\pi_1 computed in {time() - t0:.2f} seconds.")

print(f"H0 Rank: {h0.rank}")
print(f"H1 Rank: {h1.rank}")
print(f"H2 Rank: {h2.rank}")
print(f"Pi1 Generators: {len(pi1.generators)}")

H0 basis computed in 0.48 seconds.


## Visualizing the Workspace

We now pass the pre-calculated invariants to `visualize_topoview`. The UMAP projection and plotly layout are handled internally.

In [ ]:
# 4. Visualize and Save to HTML
res = visualize_topoview(
    sc, 
    dimension=3, 
    points=points_7d_torus,
    h0_basis=h0,
    h1_basis=h1,
    h2_basis=h2,
    pi1_result=pi1,
    title="7D Torus ($T^2$) Topological Workspace",
    features=["curvature", "dual", "components"],
    output_mode="full"
)
res.write_html("TopoView-7D-Torus.html")